# Exploración de archivos XBRL — ISA

Este notebook tiene como objetivo explorar los estados financieros
reportados por ISA en formato XBRL y determinar:

1. La estructura de los archivos.
2. Los conceptos XBRL correspondientes a las variables financieras de interés.
3. Las unidades y fechas asociadas a cada dato.
4. El tratamiento necesario para convertir información YTD en datos trimestrales.

Las variables de interés son:

- Ingresos
- EBITDA
- Deuda corriente
- Deuda no corriente
- Gastos financieros
- Caja y equivalentes de efectivo

Este notebook corresponde a una etapa exploratoria. La extracción
automatizada y la construcción del panel definitivo se realizarán posteriormente.

In [21]:
import collections
import collections.abc

# Compatibilidad de Arelle con versiones modernas de Python
for _cls in [
    "MutableSet", "MutableMapping", "Mapping", "Sequence",
    "Callable", "Iterable", "Container", "Hashable",
    "Sized", "Set", "MutableSequence"
]:
    if not hasattr(collections, _cls) and hasattr(collections.abc, _cls):
        setattr(collections, _cls, getattr(collections.abc, _cls))

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np

from arelle import Cntlr

In [15]:
PROJECT_ROOT = Path.cwd().parent

RAW_ISA_PATH = PROJECT_ROOT / "Data" / "raw" / "ISA"

list(RAW_ISA_PATH.iterdir())[:10]

[WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2021Q1_2021-03-31.xbrl'),
 WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2021Q2_2021-06-30.xbrl'),
 WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2021Q3_2021-09-30.xbrl'),
 WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2021Q4_2021-12-31.xbrl'),
 WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2022Q1_2022-03-31.xbrl'),
 WindowsPath('c:/Users/57316/Documents/SIMON MONSALVE/UNIVERSIDAD/OCTAVO SEMESTRE/credit_risk_estimations/credit-risk-colombia/Data/raw/ISA/2022Q2_2022-06-30.xbrl'),
 Win

In [4]:
print("Directorio actual:", Path.cwd())
print("Ruta ISA:", RAW_ISA_PATH)

Directorio actual: c:\Users\57316\Documents\SIMON MONSALVE\UNIVERSIDAD\OCTAVO SEMESTRE\credit_risk_estimations\credit-risk-colombia\notebooks
Ruta ISA: c:\Users\57316\Documents\SIMON MONSALVE\UNIVERSIDAD\OCTAVO SEMESTRE\credit_risk_estimations\credit-risk-colombia\data\raw\ISA


In [5]:
xbrl_files = sorted(RAW_ISA_PATH.glob("*.xbrl"))

print(f"Archivos encontrados: {len(xbrl_files)}")

for file in xbrl_files:
    print(file.name)

Archivos encontrados: 22
2021Q1_2021-03-31.xbrl
2021Q2_2021-06-30.xbrl
2021Q3_2021-09-30.xbrl
2021Q4_2021-12-31.xbrl
2022Q1_2022-03-31.xbrl
2022Q2_2022-06-30.xbrl
2022Q3_2022-09-30.xbrl
2022Q4_2022-12-31.xbrl
2023Q1_2023-03-31.xbrl
2023Q2_2023-03-31.xbrl
2023Q3_2023-09-30.xbrl
2023Q4_2023-12-31.xbrl
2024Q1_2024-03-31.xbrl
2024Q2_2024-06-30.xbrl
2024Q3_2024-09-30.xbrl
2024Q4_2024-12-31.xbrl
2025Q1_2025-03-31.xbrl
2025Q2_2025-06-30.xbrl
2025Q3_2025-09-30.xbrl
2025Q4_2025-12-31.xbrl
2026Q1_2026-03-31.xbrl
2026Q6_2026-06-30.xbrl


In [6]:
file_path = xbrl_files[0]

print(file_path)

cntlr = Cntlr.Cntlr()

model_xbrl = cntlr.modelManager.load(str(file_path))

print("Archivo cargado correctamente")
print("Número de facts:", len(list(model_xbrl.facts)))

c:\Users\57316\Documents\SIMON MONSALVE\UNIVERSIDAD\OCTAVO SEMESTRE\credit_risk_estimations\credit-risk-colombia\data\raw\ISA\2021Q1_2021-03-31.xbrl
Archivo cargado correctamente
Número de facts: 6151


In [7]:
facts = []

for fact in model_xbrl.facts:

    context = fact.context

    facts.append({
        "name": str(fact.concept.qname) if fact.concept is not None else None,
        "value": fact.value,
        "isNumeric": fact.isNumeric,
        "contextID": fact.contextID,
        "unitID": fact.unitID,
        "startDate": context.startDatetime,
        "endDate": context.endDatetime,
        "isInstant": context.isInstantPeriod,
        "isStartEnd": context.isStartEndPeriod,
    })

facts_df = pd.DataFrame(facts)

facts_df.head()

,name,value,isNumeric,contextID,unitID,startDate,endDate,isInstant,isStartEnd
0,co-sfc-core:ActividadPrincipal,"""ISA tiene por objeto :\n- La prestación del s...",False,TrimestreAcumuladoActual,NaN,2021-01-01,2021-04-01,False,True
1,co-sfc-core:ActivosFinancierosCorrientesAlCost...,541401202,True,SaldoActualInicio,COP,NaT,2021-01-01,True,False
2,co-sfc-core:ActivosFinancierosCorrientesAlCost...,797012,True,SaldoActualInicio,COP,NaT,2021-01-01,True,False
3,co-sfc-core:ActivosFinancierosCorrientesValorR...,617764328,True,CierreTrimestreActual,COP,NaT,2021-04-01,True,False
4,co-sfc-core:ActivosFinancierosCorrientesValorR...,541401202,True,SaldoActualInicio,COP,NaT,2021-01-01,True,False


In [8]:
# Conceptos candidatos que queremos inspeccionar
candidate_tags = [
    "ifrs:Revenue",
    "ifrs:RevenueAndOperatingIncome",
    "ifrs:Cash",
    "ifrs:CashAndCashEquivalents",
    "ifrs:CurrentBorrowingsAndCurrentPortionOfNoncurrentLoansReceived",
    "ifrs:ShorttermBorrowings",
    "ifrs:Borrowings",
    "ifrs:LongtermBorrowings",
    "ifrs:FinanceCosts",
    "ifrs:FinanceIncomeCost",
    "ifrs:InterestExpenseOnBorrowings",
    "ifrs:DepreciationAndAmortisationExpense",
    "ifrs:DepreciationExpense",
    "ifrs:AmortisationExpense"
]

candidate_facts = facts_df[
    facts_df["name"].isin(candidate_tags)
].copy()

candidate_facts.sort_values(
    ["name", "endDate"]
)

,name,value,isNumeric,contextID,unitID,startDate,endDate,isInstant,isStartEnd
488,ifrs:AmortisationExpense,980178,True,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
487,ifrs:AmortisationExpense,808334,True,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True
566,ifrs:Borrowings,4822107379,True,SaldoActualInicio,COP,NaT,2021-01-01,True,False
565,ifrs:Borrowings,4820890122,True,CierreTrimestreActual,COP,NaT,2021-04-01,True,False
570,ifrs:Cash,539123548,True,SaldoActualInicio,COP,NaT,2021-01-01,True,False
569,ifrs:Cash,617749088,True,CierreTrimestreActual,COP,NaT,2021-04-01,True,False
576,ifrs:CashAndCashEquivalents,541370776,True,SaldoAnteriorInicio,COP,NaT,2020-01-01,True,False
331,ifrs:CashAndCashEquivalents,1519026,True,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,COP,NaT,2020-04-01,True,False
333,ifrs:CashAndCashEquivalents,1519026,True,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,COP,NaT,2020-04-01,True,False
335,ifrs:CashAndCashEquivalents,1519026,True,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,COP,NaT,2020-04-01,True,False


In [9]:
pd.set_option("display.max_rows", 200)

concepts = (
    facts_df["name"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(f"Número de conceptos únicos: {len(concepts)}")

concepts.to_frame().head(100)

concepts[
    concepts.str.contains(
        "Revenue|Income|Cash|Debt|Borrow|Interest|Operating|Depreciation|Amort",
        case=False,
        regex=True
    )
]

Número de conceptos únicos: 985


1       co-sfc-core:ActivosFinancierosCorrientesAlCost...
2       co-sfc-core:ActivosFinancierosCorrientesAlCost...
10             co-sfc-core:AjusteAlCostoAmortizadoCartera
13      co-sfc-core:AjusteAlCostoAmortizadoOtrosActivo...
31      co-sfc-core:AjusteCostoAmortizadoPasivosFinanc...
423              ifrs:AccumulatedOtherComprehensiveIncome
487                              ifrs:AmortisationExpense
489     ifrs:AmortisationIntangibleAssetsOtherThanGood...
551     ifrs:BankingArrangementsClassifiedAsCashEquiva...
565                                       ifrs:Borrowings
569                                             ifrs:Cash
571     ifrs:CashAdvancesAndLoansMadeToOtherPartiesCla...
330                           ifrs:CashAndCashEquivalents
577                                  ifrs:CashEquivalents
579     ifrs:CashFlowsFromLosingControlOfSubsidiariesO...
581           ifrs:CashFlowsFromUsedInFinancingActivities
583           ifrs:CashFlowsFromUsedInInvestingActivities
585           

## Identificación de conceptos financieros

A continuación se inspeccionan los conceptos XBRL potencialmente
relacionados con las variables requeridas para el análisis.

In [10]:
keywords = {
    "ingresos": "revenue|income",
    "caja": "cash|cash.*equivalent",
    "deuda": "debt|borrow|loan",
    "intereses": "interest|finance",
    "depreciacion": "depreciation",
    "amortizacion": "amortisation|amortization",
}

for variable, pattern in keywords.items():
    print(f"\n### {variable.upper()}")
    result = concepts[
        concepts.str.contains(pattern, case=False, regex=True, na=False)
    ]
    print(result.to_string(index=False))


### INGRESOS
          ifrs:AccumulatedOtherComprehensiveIncome
                          ifrs:ComprehensiveIncome
ifrs:ComprehensiveIncomeAttributableToNoncontro...
ifrs:ComprehensiveIncomeAttributableToOwnersOfP...
                      ifrs:CurrentTaxExpenseIncome
ifrs:CurrentTaxExpenseIncomeAndAdjustmentsForCu...
ifrs:DeferredTaxExpenseIncomeRecognisedInProfit...
ifrs:DeferredTaxExpenseIncomeRelatingToOriginat...
ifrs:DescriptionOfAccountingPolicyForFeeAndComm...
ifrs:DescriptionOfAccountingPolicyForFinanceInc...
ifrs:DescriptionOfAccountingPolicyForIncomeTaxE...
ifrs:DescriptionOfAccountingPolicyForInterestIn...
ifrs:DescriptionOfAccountingPolicyForRecognitio...
ifrs:DescriptionOfAccountingPolicyForTradingInc...
ifrs:DisclosureOfAnalysisOfOtherComprehensiveIn...
        ifrs:DisclosureOfDeferredIncomeExplanatory
ifrs:DisclosureOfFeeAndCommissionIncomeExpenseE...
  ifrs:DisclosureOfFinanceIncomeExpenseExplanatory
         ifrs:DisclosureOfFinanceIncomeExplanatory
             ifrs

In [11]:
for tag in candidate_tags:

    subset = facts_df[facts_df["name"] == tag].copy()

    if not subset.empty:
        print("\n" + "=" * 100)
        print(tag)
        display(
            subset[
                [
                    "name",
                    "value",
                    "contextID",
                    "unitID",
                    "startDate",
                    "endDate",
                    "isInstant",
                    "isStartEnd"
                ]
            ].sort_values(["endDate", "startDate"])
        )


ifrs:Revenue


,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
5952,ifrs:Revenue,308743699,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
5947,ifrs:Revenue,1239484523.00,AnualAnterior_ifrs_DisclosureOfSignificantInve...,COP,2020-01-01,2021-01-01,False,True
5948,ifrs:Revenue,1239484523,AnualAnterior_ifrs_DisclosureOfSignificantInve...,COP,2020-01-01,2021-01-01,False,True
5949,ifrs:Revenue,332401870.00,Periodo_ifrs_DisclosureOfSignificantInvestment...,COP,2021-01-01,2021-04-01,False,True
5950,ifrs:Revenue,332401870,Periodo_ifrs_DisclosureOfSignificantInvestment...,COP,2021-01-01,2021-04-01,False,True
5951,ifrs:Revenue,332401870,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True



ifrs:RevenueAndOperatingIncome


,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
5954,ifrs:RevenueAndOperatingIncome,308743699,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
5953,ifrs:RevenueAndOperatingIncome,332401870,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True



ifrs:Cash


...

,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
1272,ifrs:DepreciationAndAmortisationExpense,43761698,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
1271,ifrs:DepreciationAndAmortisationExpense,48654789,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True



ifrs:DepreciationExpense


,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
1274,ifrs:DepreciationExpense,42781520,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
1273,ifrs:DepreciationExpense,47846455,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True



ifrs:AmortisationExpense


,name,value,contextID,unitID,startDate,endDate,isInstant,isStartEnd
488,ifrs:AmortisationExpense,980178,TrimestreAcumuladoAnterior,COP,2020-01-01,2020-04-01,False,True
487,ifrs:AmortisationExpense,808334,TrimestreAcumuladoActual,COP,2021-01-01,2021-04-01,False,True


In [12]:
contexts = []

for context_id, context in model_xbrl.contexts.items():

    contexts.append({
        "contextID": context_id,
        "startDate": context.startDatetime,
        "endDate": context.endDatetime,
        "isInstant": context.isInstantPeriod,
        "isStartEnd": context.isStartEndPeriod,
    })

contexts_df = pd.DataFrame(contexts)

contexts_df.sort_values(
    ["endDate", "startDate"]
).head(50)

,contextID,startDate,endDate,isInstant,isStartEnd
57,AnualPrevioAnterior_ifrs_DisclosureOfComparati...,2019-01-01,2020-01-01,False,True
58,AnualPrevioAnterior_ifrs_DisclosureOfComparati...,2019-01-01,2020-01-01,False,True
59,AnualPrevioAnterior_ifrs_DisclosureOfComparati...,2019-01-01,2020-01-01,False,True
978,SaldoAnteriorInicio,NaT,2020-01-01,True,False
990,SaldoInicio_ifrs_DisclosureOfClassesOfShareCap...,NaT,2020-01-01,True,False
992,SaldoInicio_ifrs_DisclosureOfClassesOfShareCap...,NaT,2020-01-01,True,False
993,SaldoInicio_ifrs_DisclosureOfComparativeInform...,NaT,2020-01-01,True,False
994,SaldoInicio_ifrs_DisclosureOfComparativeInform...,NaT,2020-01-01,True,False
995,SaldoInicio_ifrs_DisclosureOfComparativeInform...,NaT,2020-01-01,True,False
999,SaldoInicio_ifrs_DisclosureOfIntangibleAssetsT...,NaT,2020-01-01,True,False


In [13]:
contexts_df[
    contexts_df["endDate"].dt.year == 2021
].sort_values(
    ["endDate", "startDate"]
)

,contextID,startDate,endDate,isInstant,isStartEnd
0,AnualAnterior_ifrs_DisclosureOfClassesOfShareC...,2020-01-01,2021-01-01,False,True
1,AnualAnterior_ifrs_DisclosureOfClassesOfShareC...,2020-01-01,2021-01-01,False,True
2,AnualAnterior_ifrs_DisclosureOfClassesOfShareC...,2020-01-01,2021-01-01,False,True
3,AnualAnterior_ifrs_DisclosureOfHedgeAccounting...,2020-01-01,2021-01-01,False,True
4,AnualAnterior_ifrs_DisclosureOfInvestmentPrope...,2020-01-01,2021-01-01,False,True
...,...,...,...,...,...
967,Saldo_ifrs_StatementOfChangesInEquityTable_Tre...,NaT,2021-04-01,True,False
969,Saldo_ifrs_StatementOfChangesInEquityTable_Tre...,NaT,2021-04-01,True,False
971,Saldo_ifrs_StatementOfChangesInEquityTable_Tre...,NaT,2021-04-01,True,False
973,Saldo_ifrs_StatementOfChangesInEquityTable_Tre...,NaT,2021-04-01,True,False


## Validación de conceptos financieros

In [14]:
validation_tags = {
    "Ingresos": "ifrs:Revenue",
    "Caja": "ifrs:CashAndCashEquivalents",
    "Deuda CP": "ifrs:ShorttermBorrowings",
    "Deuda LP": "ifrs:LongtermBorrowings",
    "Deuda Total": "ifrs:Borrowings",
    "Gastos financieros": "ifrs:FinanceCosts",
    "Intereses deuda": "ifrs:InterestExpenseOnBorrowings",
    "Resultado operativo": "ifrs:ProfitLossFromOperatingActivities",
    "Depreciación": "ifrs:DepreciationExpense",
    "Amortización": "ifrs:AmortisationExpense",
    "D&A": "ifrs:DepreciationAndAmortisationExpense",
}

validation_rows = []

for variable, tag in validation_tags.items():

    subset = facts_df[facts_df["name"] == tag].copy()

    for _, row in subset.iterrows():
        validation_rows.append({
            "variable": variable,
            "tag": tag,
            "value": row["value"],
            "unitID": row["unitID"],
            "contextID": row["contextID"],
            "startDate": row["startDate"],
            "endDate": row["endDate"],
            "isInstant": row["isInstant"],
            "isStartEnd": row["isStartEnd"],
        })

validation_df = pd.DataFrame(validation_rows)

validation_df.sort_values(
    ["variable", "endDate", "startDate"]
)

,variable,tag,value,unitID,contextID,startDate,endDate,isInstant,isStartEnd
31,Amortización,ifrs:AmortisationExpense,980178,COP,TrimestreAcumuladoAnterior,2020-01-01,2020-04-01,False,True
30,Amortización,ifrs:AmortisationExpense,808334,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,True
15,Caja,ifrs:CashAndCashEquivalents,541370776,COP,SaldoAnteriorInicio,NaT,2020-01-01,True,False
7,Caja,ifrs:CashAndCashEquivalents,1519026,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,True,False
9,Caja,ifrs:CashAndCashEquivalents,1519026,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,True,False
11,Caja,ifrs:CashAndCashEquivalents,1519026,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,True,False
13,Caja,ifrs:CashAndCashEquivalents,409376009,COP,CierreTrimestreAnterior,NaT,2020-04-01,True,False
14,Caja,ifrs:CashAndCashEquivalents,542198214,COP,SaldoActualInicio,NaT,2021-01-01,True,False
6,Caja,ifrs:CashAndCashEquivalents,955610,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2021-04-01,True,False
8,Caja,ifrs:CashAndCashEquivalents,955610,COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2021-04-01,True,False


## Validación detallada del archivo 2021Q1

Se inspeccionan únicamente los once tags solicitados. La consolidación se identifica por la ausencia de dimensiones XBRL en el contexto; los facts dimensionales se conservan para detectar duplicidades, pero no se mezclan con el fact consolidado.

Arelle representa el extremo final del intervalo como fecha exclusiva: `2021-04-01` corresponde al trimestre cerrado el 31 de marzo de 2021.

In [16]:
validation_tags_exact = [
    "ifrs:Revenue",
    "ifrs:CashAndCashEquivalents",
    "ifrs:ShorttermBorrowings",
    "ifrs:LongtermBorrowings",
    "ifrs:Borrowings",
    "ifrs:FinanceCosts",
    "ifrs:InterestExpenseOnBorrowings",
    "ifrs:ProfitLossFromOperatingActivities",
    "ifrs:DepreciationExpense",
    "ifrs:AmortisationExpense",
    "ifrs:DepreciationAndAmortisationExpense",
]

q1_path = RAW_ISA_PATH / "2021Q1_2021-03-31.xbrl"
q1_model = cntlr.modelManager.load(str(q1_path))

variable_by_tag = {
    "ifrs:Revenue": "ingresos",
    "ifrs:CashAndCashEquivalents": "caja",
    "ifrs:ShorttermBorrowings": "deuda_cp",
    "ifrs:LongtermBorrowings": "deuda_lp",
    "ifrs:Borrowings": "deuda_total",
    "ifrs:FinanceCosts": "gastos_financieros",
    "ifrs:InterestExpenseOnBorrowings": "intereses_deuda",
    "ifrs:ProfitLossFromOperatingActivities": "resultado_operativo",
    "ifrs:DepreciationExpense": "depreciacion",
    "ifrs:AmortisationExpense": "amortizacion",
    "ifrs:DepreciationAndAmortisationExpense": "depreciacion_amortizacion",
}

def fact_unit(fact):
    if fact.unit is None:
        return fact.unitID
    numerator, denominator = fact.unit.measures
    numerator_text = " * ".join(str(measure) for measure in numerator)
    denominator_text = " * ".join(str(measure) for measure in denominator)
    return numerator_text if not denominator_text else f"{numerator_text} / {denominator_text}"

def context_dimensions(context):
    return "; ".join(
        f"{dimension}={dimension_value.memberQname}"
        for dimension, dimension_value in context.qnameDims.items()
    )

detail_rows = []
for fact in q1_model.facts:
    tag = str(fact.concept.qname) if fact.concept is not None else None
    if tag not in validation_tags_exact:
        continue
    context = fact.context
    dimensions = context_dimensions(context)
    detail_rows.append({
        "variable": variable_by_tag[tag],
        "tag": tag,
        "valor": fact.value,
        "unitID": fact.unitID,
        "unidad": fact_unit(fact),
        "contextID": fact.contextID,
        "fecha_inicial": context.startDatetime,
        "fecha_final": context.endDatetime,
        "tipo": "instantaneo" if context.isInstantPeriod else "duracion",
        "dimensiones": dimensions or "sin dimensiones",
        "consolidado": not bool(dimensions),
        "ytd": (
            not context.isInstantPeriod
            and not dimensions
            and context.startDatetime is not None
            and context.startDatetime.month == 1
            and context.startDatetime.day == 1
        ),
    })

q1_facts_df = (
    pd.DataFrame(detail_rows)
    .sort_values(["tag", "fecha_final", "contextID"])
    .reset_index(drop=True)
)
display(q1_facts_df)
print(f"Facts encontrados: {len(q1_facts_df)}")

,variable,tag,valor,unitID,unidad,contextID,fecha_inicial,fecha_final,tipo,dimensiones,consolidado,ytd
0,amortizacion,ifrs:AmortisationExpense,980178,COP,iso4217:COP,TrimestreAcumuladoAnterior,2020-01-01,2020-04-01,duracion,sin dimensiones,True,True
1,amortizacion,ifrs:AmortisationExpense,808334,COP,iso4217:COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,duracion,sin dimensiones,True,True
2,deuda_total,ifrs:Borrowings,4822107379,COP,iso4217:COP,SaldoActualInicio,NaT,2021-01-01,instantaneo,sin dimensiones,True,False
3,deuda_total,ifrs:Borrowings,4820890122,COP,iso4217:COP,CierreTrimestreActual,NaT,2021-04-01,instantaneo,sin dimensiones,True,False
4,caja,ifrs:CashAndCashEquivalents,541370776,COP,iso4217:COP,SaldoAnteriorInicio,NaT,2020-01-01,instantaneo,sin dimensiones,True,False
5,caja,ifrs:CashAndCashEquivalents,409376009,COP,iso4217:COP,CierreTrimestreAnterior,NaT,2020-04-01,instantaneo,sin dimensiones,True,False
6,caja,ifrs:CashAndCashEquivalents,1519026,COP,iso4217:COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,instantaneo,co-sfc-core:InformacionRevelarNegociosConjunto...,False,False
7,caja,ifrs:CashAndCashEquivalents,1519026,COP,iso4217:COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,instantaneo,co-sfc-core:InformacionRevelarNegociosConjunto...,False,False
8,caja,ifrs:CashAndCashEquivalents,1519026,COP,iso4217:COP,Saldo_co-sfc-core_InformacionRevelarNegociosCo...,NaT,2020-04-01,instantaneo,co-sfc-core:InformacionRevelarNegociosConjunto...,False,False
9,caja,ifrs:CashAndCashEquivalents,542198214,COP,iso4217:COP,SaldoActualInicio,NaT,2021-01-01,instantaneo,sin dimensiones,True,False


Facts encontrados: 34


In [17]:
def one_consolidated_value(tag, context_id):
    values = q1_facts_df.loc[
        (q1_facts_df["tag"] == tag)
        & (q1_facts_df["contextID"] == context_id)
        & q1_facts_df["consolidado"],
        "valor",
    ]
    return float(values.iloc[0]) if len(values) == 1 else None

def count_consolidated(tag, context_id):
    return len(q1_facts_df.loc[
        (q1_facts_df["tag"] == tag)
        & (q1_facts_df["contextID"] == context_id)
        & q1_facts_df["consolidado"]
    ])

revenue = one_consolidated_value("ifrs:Revenue", "TrimestreAcumuladoActual")
finance_costs = one_consolidated_value("ifrs:FinanceCosts", "TrimestreAcumuladoActual")
shortterm_debt = one_consolidated_value("ifrs:ShorttermBorrowings", "CierreTrimestreActual")
longterm_debt = one_consolidated_value("ifrs:LongtermBorrowings", "CierreTrimestreActual")
borrowings = one_consolidated_value("ifrs:Borrowings", "CierreTrimestreActual")
operating_profit = one_consolidated_value("ifrs:ProfitLossFromOperatingActivities", "TrimestreAcumuladoActual")
depreciation = one_consolidated_value("ifrs:DepreciationExpense", "TrimestreAcumuladoActual")
amortisation = one_consolidated_value("ifrs:AmortisationExpense", "TrimestreAcumuladoActual")
depreciation_amortisation = one_consolidated_value("ifrs:DepreciationAndAmortisationExpense", "TrimestreAcumuladoActual")

ebitda_separate = (
    operating_profit + depreciation + amortisation
    if None not in [operating_profit, depreciation, amortisation]
    else None
)
ebitda_aggregate = (
    operating_profit + depreciation_amortisation
    if None not in [operating_profit, depreciation_amortisation]
    else None
)

summary_rows = []
for _, row in q1_facts_df[q1_facts_df["consolidado"]].iterrows():
    if row["contextID"] not in ["TrimestreAcumuladoActual", "CierreTrimestreActual"]:
        continue
    summary_rows.append({
        "variable": row["variable"],
        "tag": row["tag"],
        "valor": row["valor"],
        "tipo": row["tipo"],
        "contexto": row["contextID"],
        "conclusion": "flujo YTD consolidado" if row["ytd"] else "stock instantaneo consolidado" if row["tipo"] == "instantaneo" else "flujo consolidado",
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print("1. Consolidación: se consideran consolidados los facts sin dimensiones; los facts dimensionales aparecen separados en q1_facts_df.")
print("2. Flujos YTD: Revenue, FinanceCosts, InterestExpenseOnBorrowings, ProfitLossFromOperatingActivities, DepreciationExpense, AmortisationExpense y DepreciationAndAmortisationExpense.")
print("   Stocks instantáneos: CashAndCashEquivalents, ShorttermBorrowings, LongtermBorrowings y Borrowings.")
print(f"3. Revenue: exactamente un consolidado YTD = {count_consolidated('ifrs:Revenue', 'TrimestreAcumuladoActual') == 1}.")
print(f"   FinanceCosts: exactamente un consolidado YTD = {count_consolidated('ifrs:FinanceCosts', 'TrimestreAcumuladoActual') == 1}.")
print(f"4. ShorttermBorrowings y LongtermBorrowings: exactamente un saldo consolidado al cierre = {shortterm_debt is not None and longterm_debt is not None}.")
print(f"5. Borrowings = ShorttermBorrowings + LongtermBorrowings = {borrowings == shortterm_debt + longterm_debt} ({borrowings} = {shortterm_debt} + {longterm_debt}).")
print(f"6. EBITDA separado = {ebitda_separate}; EBITDA agregado = {ebitda_aggregate}; mismo resultado = {ebitda_separate == ebitda_aggregate}.")
print("7. Inconsistencias: Revenue y CashAndCashEquivalents incluyen facts dimensionales; no deben sumarse al fact consolidado sin dimensiones.")

,variable,tag,valor,tipo,contexto,conclusion
0,amortizacion,ifrs:AmortisationExpense,808334,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
1,deuda_total,ifrs:Borrowings,4820890122,instantaneo,CierreTrimestreActual,stock instantaneo consolidado
2,caja,ifrs:CashAndCashEquivalents,617764328,instantaneo,CierreTrimestreActual,stock instantaneo consolidado
3,depreciacion_amortizacion,ifrs:DepreciationAndAmortisationExpense,48654789,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
4,depreciacion,ifrs:DepreciationExpense,47846455,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
5,gastos_financieros,ifrs:FinanceCosts,63440268,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
6,intereses_deuda,ifrs:InterestExpenseOnBorrowings,16901559,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
7,deuda_lp,ifrs:LongtermBorrowings,4655385523,instantaneo,CierreTrimestreActual,stock instantaneo consolidado
8,resultado_operativo,ifrs:ProfitLossFromOperatingActivities,235022948,duracion,TrimestreAcumuladoActual,flujo YTD consolidado
9,ingresos,ifrs:Revenue,332401870,duracion,TrimestreAcumuladoActual,flujo YTD consolidado


1. Consolidación: se consideran consolidados los facts sin dimensiones; los facts dimensionales aparecen separados en q1_facts_df.
2. Flujos YTD: Revenue, FinanceCosts, InterestExpenseOnBorrowings, ProfitLossFromOperatingActivities, DepreciationExpense, AmortisationExpense y DepreciationAndAmortisationExpense.
   Stocks instantáneos: CashAndCashEquivalents, ShorttermBorrowings, LongtermBorrowings y Borrowings.
3. Revenue: exactamente un consolidado YTD = True.
   FinanceCosts: exactamente un consolidado YTD = True.
4. ShorttermBorrowings y LongtermBorrowings: exactamente un saldo consolidado al cierre = True.
5. Borrowings = ShorttermBorrowings + LongtermBorrowings = True (4820890122.0 = 165504599.0 + 4655385523.0).
6. EBITDA separado = 283677737.0; EBITDA agregado = 283677737.0; mismo resultado = True.
7. Inconsistencias: Revenue y CashAndCashEquivalents incluyen facts dimensionales; no deben sumarse al fact consolidado sin dimensiones.


## Validación temporal de conceptos — ISA 2021

Se reutilizan los mismos once tags validados en Q1. Los flujos se seleccionan desde el contexto consolidado YTD actual y los stocks desde el contexto consolidado instantáneo de cierre.

In [24]:
files_2021 = sorted(RAW_ISA_PATH.glob("2021Q*.xbrl"))
print("Archivos 2021:")
for file in files_2021:
    print(file.name)

flow_tags = [
    "ifrs:Revenue",
    "ifrs:FinanceCosts",
    "ifrs:InterestExpenseOnBorrowings",
    "ifrs:ProfitLossFromOperatingActivities",
    "ifrs:DepreciationExpense",
    "ifrs:AmortisationExpense",
    "ifrs:DepreciationAndAmortisationExpense",
]
stock_tags = [
    "ifrs:CashAndCashEquivalents",
    "ifrs:ShorttermBorrowings",
    "ifrs:LongtermBorrowings",
    "ifrs:Borrowings",
]
all_temporal_tags = validation_tags_exact

period_rows = []
validation_counts = []

for file in files_2021:
    period = file.name.split("_")[0]
    period_model = cntlr.modelManager.load(str(file))
    facts_by_tag = {tag: [] for tag in all_temporal_tags}

    for fact in period_model.facts:
        tag = str(fact.concept.qname) if fact.concept is not None else None
        if tag not in facts_by_tag:
            continue
        context = fact.context
        dimensions = context_dimensions(context)
        if dimensions:
            continue
        facts_by_tag[tag].append({
            "tag": tag,
            "valor": float(fact.value),
            "unitID": fact.unitID,
            "contextID": fact.contextID,
            "fecha_inicial": context.startDatetime,
            "fecha_final": context.endDatetime,
            "instantaneo": context.isInstantPeriod,
        })

    for tag in all_temporal_tags:
        current_candidates = [
            row for row in facts_by_tag[tag]
            if row["contextID"] in ["TrimestreAcumuladoActual", "CierreTrimestreActual"]
        ]
        if tag in flow_tags:
            selected = [
                row for row in current_candidates
                if not row["instantaneo"]
                and row["fecha_inicial"] is not None
                and row["fecha_inicial"].month == 1
                and row["fecha_inicial"].day == 1
            ]
        else:
            selected = [
                row for row in current_candidates
                if row["instantaneo"]
                and row["contextID"] == "CierreTrimestreActual"
            ]
        validation_counts.append({
            "periodo": period,
            "tag": tag,
            "candidatos_consolidados": len(selected),
        })
        if len(selected) == 1:
            row = selected[0].copy()
            row["periodo"] = period
            period_rows.append(row)

    cntlr.modelManager.close(period_model)

validation_counts_df = pd.DataFrame(validation_counts)
temporal_facts_df = pd.DataFrame(period_rows)

print("Conteos de candidatos consolidados seleccionables:")
display(validation_counts_df.pivot(index="periodo", columns="tag", values="candidatos_consolidados"))
print("Facts seleccionados:")
display(temporal_facts_df.sort_values(["periodo", "tag"]))

Archivos 2021:
2021Q1_2021-03-31.xbrl
2021Q2_2021-06-30.xbrl
2021Q3_2021-09-30.xbrl
2021Q4_2021-12-31.xbrl
Conteos de candidatos consolidados seleccionables:


tag,ifrs:AmortisationExpense,ifrs:Borrowings,ifrs:CashAndCashEquivalents,ifrs:DepreciationAndAmortisationExpense,ifrs:DepreciationExpense,ifrs:FinanceCosts,ifrs:InterestExpenseOnBorrowings,ifrs:LongtermBorrowings,ifrs:ProfitLossFromOperatingActivities,ifrs:Revenue,ifrs:ShorttermBorrowings
periodo,,,,,,,,,,,
2021Q1,1,1,1,1,1,1,1,1,1,1,1
2021Q2,1,1,1,1,1,1,1,1,1,1,1
2021Q3,1,1,1,1,1,1,1,1,1,1,1
2021Q4,1,1,1,1,1,1,1,1,1,1,1


Facts seleccionados:


,tag,valor,unitID,contextID,fecha_inicial,fecha_final,instantaneo,periodo
9,ifrs:AmortisationExpense,8.083340e+05,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
4,ifrs:Borrowings,4.820890e+09,COP,CierreTrimestreActual,NaT,2021-04-01,True,2021Q1
1,ifrs:CashAndCashEquivalents,6.177643e+08,COP,CierreTrimestreActual,NaT,2021-04-01,True,2021Q1
10,ifrs:DepreciationAndAmortisationExpense,4.865479e+07,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
8,ifrs:DepreciationExpense,4.784646e+07,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
5,ifrs:FinanceCosts,6.344027e+07,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
6,ifrs:InterestExpenseOnBorrowings,1.690156e+07,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
3,ifrs:LongtermBorrowings,4.655386e+09,COP,CierreTrimestreActual,NaT,2021-04-01,True,2021Q1
7,ifrs:ProfitLossFromOperatingActivities,2.350229e+08,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1
0,ifrs:Revenue,3.324019e+08,COP,TrimestreAcumuladoActual,2021-01-01,2021-04-01,False,2021Q1


In [25]:
def selected_value(period, tag):
    values = temporal_facts_df.loc[
        (temporal_facts_df["periodo"] == period)
        & (temporal_facts_df["tag"] == tag),
        "valor",
    ]
    return float(values.iloc[0]) if len(values) == 1 else np.nan

periods = [f"2021Q{quarter}" for quarter in range(1, 5)]
ytd_rows = []

for period in periods:
    ytd_rows.append({
        "periodo": period,
        "ingresos_ytd": selected_value(period, "ifrs:Revenue"),
        "finance_costs_ytd": selected_value(period, "ifrs:FinanceCosts"),
        "resultado_operativo_ytd": selected_value(period, "ifrs:ProfitLossFromOperatingActivities"),
        "d_and_a_ytd": selected_value(period, "ifrs:DepreciationAndAmortisationExpense"),
        "caja": selected_value(period, "ifrs:CashAndCashEquivalents"),
        "deuda_cp": selected_value(period, "ifrs:ShorttermBorrowings"),
        "deuda_lp": selected_value(period, "ifrs:LongtermBorrowings"),
    })

ytd_df = pd.DataFrame(ytd_rows).set_index("periodo")
print("Tabla temporal: flujos YTD y stocks al cierre")
display(ytd_df)

flow_ytd_columns = [
    "ingresos_ytd",
    "finance_costs_ytd",
    "resultado_operativo_ytd",
    "d_and_a_ytd",
]
flow_quarterly_df = ytd_df[flow_ytd_columns].diff()
flow_quarterly_df.iloc[0] = ytd_df.iloc[0][flow_ytd_columns]
flow_quarterly_df = flow_quarterly_df.rename(columns={
    "ingresos_ytd": "ingresos_trimestrales",
    "finance_costs_ytd": "finance_costs_trimestrales",
    "resultado_operativo_ytd": "resultado_operativo_trimestral",
    "d_and_a_ytd": "d_and_a_trimestral",
})
flow_quarterly_df["ebitda_trimestral"] = (
    flow_quarterly_df["resultado_operativo_trimestral"]
    + flow_quarterly_df["d_and_a_trimestral"]
)

print("Tabla de flujos trimestrales derivados por diferencias YTD")
display(flow_quarterly_df)

borrowings_ytd_df = pd.DataFrame(index=periods)
borrowings_ytd_df["deuda_total"] = ytd_df["deuda_cp"] + ytd_df["deuda_lp"]
borrowings_ytd_df["borrowings_reportado"] = [
    selected_value(period, "ifrs:Borrowings") for period in periods
]
borrowings_ytd_df["diferencia"] = (
    borrowings_ytd_df["borrowings_reportado"]
    - borrowings_ytd_df["deuda_total"]
)
print("Control de deuda total")
display(borrowings_ytd_df)

expected_counts = validation_counts_df["candidatos_consolidados"].eq(1).all()
all_debt_controls_ok = borrowings_ytd_df["diferencia"].eq(0).all()
all_ebitda_values_available = flow_quarterly_df["ebitda_trimestral"].notna().all()

print(f"Un candidato consolidado por tag y trimestre: {expected_counts}")
print(f"Borrowings = ShorttermBorrowings + LongtermBorrowings en Q1-Q4: {all_debt_controls_ok}")
print(f"EBITDA trimestral disponible en Q1-Q4: {all_ebitda_values_available}")

assert len(files_2021) == 4
assert expected_counts
assert all_debt_controls_ok
assert all_ebitda_values_available

Tabla temporal: flujos YTD y stocks al cierre


,ingresos_ytd,finance_costs_ytd,resultado_operativo_ytd,d_and_a_ytd,caja,deuda_cp,deuda_lp
periodo,,,,,,,
2021Q1,3.324019e+08,63440268.0,2.350229e+08,48654789.0,6.177643e+08,165504599.0,4.655386e+09
2021Q2,6.751931e+08,136440412.0,4.906948e+08,98574118.0,1.492654e+09,276305988.0,4.522584e+09
2021Q3,1.026321e+09,230341240.0,7.531711e+08,148322754.0,1.205650e+09,285445444.0,4.506824e+09
2021Q4,1.385334e+09,323980235.0,1.002043e+09,200719035.0,5.458365e+08,158138806.0,4.620915e+09


Tabla de flujos trimestrales derivados por diferencias YTD


,ingresos_trimestrales,finance_costs_trimestrales,resultado_operativo_trimestral,d_and_a_trimestral,ebitda_trimestral
periodo,,,,,
2021Q1,332401870.0,63440268.0,235022948.0,48654789.0,283677737.0
2021Q2,342791251.0,73000144.0,255671844.0,49919329.0,305591173.0
2021Q3,351127612.0,93900828.0,262476286.0,49748636.0,312224922.0
2021Q4,359013297.0,93638995.0,248872135.0,52396281.0,301268416.0


Control de deuda total


,deuda_total,borrowings_reportado,diferencia
2021Q1,4.820890e+09,4.820890e+09,0.0
2021Q2,4.798890e+09,4.798890e+09,0.0
2021Q3,4.792269e+09,4.792269e+09,0.0
2021Q4,4.779054e+09,4.779054e+09,0.0


Un candidato consolidado por tag y trimestre: True
Borrowings = ShorttermBorrowings + LongtermBorrowings en Q1-Q4: True
EBITDA trimestral disponible en Q1-Q4: True
